<a href="https://colab.research.google.com/github/mahmii12/FlyRank-Assignment-/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages loaded |  declining rate (proxy label, used only to evaluate, never as an input):", round(df["is_declining"].mean(), 3))

30000 pages loaded |  declining rate (proxy label, used only to evaluate, never as an input): 0.542


## 1. Two signal checks (one bucket table each, with n)

**Signal A — staleness, behind the refresh flags.** Claim: *"Pages that haven't been updated in
180+ days are more likely to be declining."* This is the assumption FlyRank's refresh flag leans
on. Test it directly against the (proxy) decline label.

In [2]:
stale = df["days_since_last_update"] >= 180
tbl_stale = (
    df.groupby(stale.map({True: "stale (>=180d)", False: "fresh (<180d)"}))["is_declining"]
    .agg(n="count", decline_rate="mean")
)
print("Signal A — staleness vs decline rate")
print(tbl_stale)

Signal A — staleness vs decline rate
                            n  decline_rate
days_since_last_update                     
fresh (<180d)           29826      0.542480
stale (>=180d)            174      0.471264


**Verdict A: OPPOSITE.** n=174 stale pages clears the ~50-row floor. Stale pages decline
*less* often (0.47) than fresh pages (0.54) — the reverse of what the refresh-flag story assumes.
Most likely explanation: pages get updated *because* they're already declining, so "stale" mostly
selects for content nobody has bothered to touch — often stable, evergreen, or simply ignored —
not content that's actively losing ground. Staleness alone is not a decline signal here. That's
a real finding, not a failed test: it means my rule can't lean on staleness by itself, and it
explains why the score below leans on Signal B instead.

**Signal B — CTR vs. position, behind the CTR-fix logic.** Claim: *"Better average position
gets a higher CTR."* This is the assumption behind flagging a page as under-clicking for where it
ranks. Rates need proper denominators (a mean of per-page CTRs is not the true rate), so this
uses total clicks / total impressions per tier — not an average of ratios. `avg_position == 0`
means "no data," not rank zero, so those rows are dropped first.

In [3]:
has_position = df["avg_position"] > 0  # 0 means "no data", not rank zero — drop it
tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]

g = df[has_position].groupby("position_tier")
tbl_ctr = pd.DataFrame({
    "n": g.size(),
    "weighted_ctr_pct": g.apply(lambda d: d["clicks_90d"].sum() / d["impressions_90d"].sum() * 100),
}).reindex(tier_order)
print("Signal B — position tier vs impression-weighted CTR")
print(tbl_ctr)

Signal B — position tier vs impression-weighted CTR
                   n  weighted_ctr_pct
position_tier                         
top_3           1116          0.488544
page_1         11814          0.350324
striking        7304          0.346876
page_3_5        7242          0.154905
deep            1319          0.041359


/tmp/ipykernel_2113/1845519874.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  "weighted_ctr_pct": g.apply(lambda d: d["clicks_90d"].sum() / d["impressions_90d"].sum() * 100),


**Verdict B: MIXED (directionally CONFIRMED).** Every tier clears n=1,000+. The big-picture
direction holds — `deep` (0.04%) < `page_3_5` (0.16%) < `striking`/`page_1` (~0.35% each) <
`top_3` (0.49%) — worse position really does mean lower CTR on average. But it's not clean:
`striking` (position 11–20) and `page_1` (position 3–10) land at almost the same weighted CTR,
so "better position always beats worse position" isn't true tier-by-tier in the middle of the
pack. Good enough to build a per-tier CTR benchmark on, not good enough to treat position as a
precise CTR predictor.

## 2. My rule and its reason code

**In plain words:** A page is worth reviewing first if it (a) has real position data and enough
impressions to matter, and (b) is getting a lower CTR than other pages sitting in the same
position tier — i.e. it's underperforming *for where it already ranks*. That's the CTR-fix logic
from the session, encoded directly, using the per-tier benchmark from Signal B above. I deliberately
did not lean on staleness — Signal A showed that assumption doesn't hold in this data.

Score = eligible × (tier CTR benchmark − page CTR) × impressions_90d, clipped at 0 so pages that
already beat their tier's benchmark score zero. One reason code, one action label. No label-derived
or future-window inputs — `tier_benchmark`, `ctr`, `impressions_90d`, `avg_position`, and
`position_tier` are all pre-decision, current-window observed/derived signals from the data
dictionary, not product flags.

In [4]:
VISIBLE_MIN = 500  # impressions_90d floor for "enough traffic to matter"

benchmark = tbl_ctr["weighted_ctr_pct"]  # per-tier CTR benchmark from Signal B, reused as the rule's threshold
df["tier_ctr_benchmark"] = df["position_tier"].map(benchmark)

eligible = has_position & (df["impressions_90d"] >= VISIBLE_MIN)
ctr_gap = (df["tier_ctr_benchmark"] - df["ctr"]).clip(lower=0)  # 0 if page already beats its tier

df["baseline_action_score"] = np.where(eligible, ctr_gap * df["impressions_90d"], 0.0)
df["reason_code"] = np.where(df["baseline_action_score"] > 0, "ctr_below_tier_benchmark", "no_flag")
df["action_label"] = np.where(df["baseline_action_score"] > 0, "review_ctr_fix", "monitor")

queue = df.sort_values("baseline_action_score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

out_cols = ["rank", "content_id", "client_id", "baseline_action_score", "reason_code",
            "action_label", "position_tier", "avg_position", "ctr", "tier_ctr_benchmark",
            "impressions_90d", "days_since_last_update", "content_type"]

os.makedirs("work/outputs", exist_ok=True)
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Flagged rows (action != monitor):", int((queue['action_label'] == 'review_ctr_fix').sum()), "/", len(queue))
print("Wrote work/outputs/baseline_action_score.csv")
queue[out_cols].head(10)

Flagged rows (action != monitor): 11631 / 30000
Wrote work/outputs/baseline_action_score.csv


,rank,content_id,client_id,baseline_action_score,reason_code,action_label,position_tier,avg_position,ctr,tier_ctr_benchmark,impressions_90d,days_since_last_update,content_type
0,1,content_8c19996aa890,client_4e07408562,172404.141956,ctr_below_tier_benchmark,review_ctr_fix,top_3,2.5,0.15,0.488544,509252,20,keyword article
1,2,content_8451fc6f034d,client_d029fa3a95,124789.962461,ctr_below_tier_benchmark,review_ctr_fix,top_3,2.3,0.03,0.488544,272144,20,keyword article
2,3,content_5fe46e04994d,client_4e07408562,108887.742905,ctr_below_tier_benchmark,review_ctr_fix,page_1,4.2,0.14,0.350324,517715,104,keyword article
3,4,content_36ff89c8214e,client_19581e27de,88624.627778,ctr_below_tier_benchmark,review_ctr_fix,page_1,7.3,0.05,0.350324,295097,104,keyword article
4,5,content_c8e9d6ab9013,client_19581e27de,73104.852519,ctr_below_tier_benchmark,review_ctr_fix,page_1,9.7,0.00,0.350324,208678,104,keyword article
5,6,content_c84a0ab98e90,client_f369cb89fc,71518.996514,ctr_below_tier_benchmark,review_ctr_fix,page_1,7.8,0.03,0.350324,223271,20,keyword article
6,7,content_e12868d1f396,client_4e07408562,62661.039592,ctr_below_tier_benchmark,review_ctr_fix,top_3,2.9,0.07,0.488544,149712,7,keyword article
7,8,content_4a6607efcb46,client_6208ef0f77,61286.156110,ctr_below_tier_benchmark,review_ctr_fix,top_3,2.2,0.01,0.488544,128068,104,keyword article
8,9,content_cb112fce36be,client_19581e27de,58983.222991,ctr_below_tier_benchmark,review_ctr_fix,page_1,5.6,0.16,0.350324,309910,104,keyword article
9,10,content_73c54f78c06a,client_f369cb89fc,53560.013361,ctr_below_tier_benchmark,review_ctr_fix,page_1,4.7,0.10,0.350324,213963,20,keyword article


## 3. Top-10 review

For each: the action, why it's there, and what would make it wrong.

In [5]:
top10 = queue.head(10)
for _, r in top10.iterrows():
    print(f"#{int(r['rank'])} {r['content_id']} | action={r['action_label']} | reason={r['reason_code']}")
    print(f"    score={r['baseline_action_score']:.1f} | tier={r['position_tier']} (pos {r['avg_position']:.1f}) "
          f"| ctr={r['ctr']:.2f}% vs tier benchmark {r['tier_ctr_benchmark']:.2f}% | impressions_90d={int(r['impressions_90d'])}")
    print()

#1 content_8c19996aa890 | action=review_ctr_fix | reason=ctr_below_tier_benchmark
    score=172404.1 | tier=top_3 (pos 2.5) | ctr=0.15% vs tier benchmark 0.49% | impressions_90d=509252

#2 content_8451fc6f034d | action=review_ctr_fix | reason=ctr_below_tier_benchmark
    score=124790.0 | tier=top_3 (pos 2.3) | ctr=0.03% vs tier benchmark 0.49% | impressions_90d=272144

#3 content_5fe46e04994d | action=review_ctr_fix | reason=ctr_below_tier_benchmark
    score=108887.7 | tier=page_1 (pos 4.2) | ctr=0.14% vs tier benchmark 0.35% | impressions_90d=517715

#4 content_36ff89c8214e | action=review_ctr_fix | reason=ctr_below_tier_benchmark
    score=88624.6 | tier=page_1 (pos 7.3) | ctr=0.05% vs tier benchmark 0.35% | impressions_90d=295097

#5 content_c8e9d6ab9013 | action=review_ctr_fix | reason=ctr_below_tier_benchmark
    score=73104.9 | tier=page_1 (pos 9.7) | ctr=0.00% vs tier benchmark 0.35% | impressions_90d=208678

#6 content_c84a0ab98e90 | action=review_ctr_fix | reason=ctr_below_ti

**Top-10 review** (from this run — content ids will differ slightly if you re-run after data
changes, but the pattern holds):

All 10 are `keyword article` pages sitting in `top_3` or `page_1` tiers, each pulling 100k–500k+
impressions_90d, with CTR near zero against a tier benchmark around 0.35–0.49%. One line each:

1–10. **Action: review_ctr_fix.** Why it's there: ranks well (position 2–10) and gets a large,
real audience, but converts almost none of that audience into clicks relative to other pages at
the same position — the classic CTR-fix pattern. What would make it wrong: the query intent
behind these pages might be informational or navigational (a definition or brand query), where a
near-zero CTR is *expected at any position* and there's no title/snippet problem to fix — I can't
see intent from this slice, so I can't rule that out from the data alone. `main_intent` is in the
dataset and worth checking before trusting any single row here.
**What would make it wrong:** two live possibilities I can't tell apart from this data alone — either the title/snippet is genuinely weak and fixable, or the query intent behind these pages is informational/navigational, where a near-zero CTR is expected regardless of position. I'm treating both as open until someone checks main_intent and looks at the actual title/snippet for a few of these.

## 4. Weak picks + leakage check

Which picks look wrong, and why? Confirm no product flags or future-window inputs leaked in.

In [6]:
# Weak-pick scan: rows where the score is driven almost entirely by raw impression volume
# rather than a genuinely large CTR gap — i.e. the "impressions_90d" term is doing the work.
top10["ctr_gap_pct"] = top10["tier_ctr_benchmark"] - top10["ctr"]
print(top10[["rank", "content_id", "impressions_90d", "ctr_gap_pct", "content_type"]])

print()
print("Columns used in the score:", ["position_tier", "avg_position", "ctr", "impressions_90d"])
print("None of these are product flags (health_score, priority_score, action_type, refresh_tier)")
print("and none use a future window or is_declining_label — all are pre-decision, current-window fields.")

   rank            content_id  impressions_90d  ctr_gap_pct     content_type
0     1  content_8c19996aa890           509252     0.338544  keyword article
1     2  content_8451fc6f034d           272144     0.458544  keyword article
2     3  content_5fe46e04994d           517715     0.210324  keyword article
3     4  content_36ff89c8214e           295097     0.300324  keyword article
4     5  content_c8e9d6ab9013           208678     0.350324  keyword article
5     6  content_c84a0ab98e90           223271     0.320324  keyword article
6     7  content_e12868d1f396           149712     0.418544  keyword article
7     8  content_4a6607efcb46           128068     0.478544  keyword article
8     9  content_cb112fce36be           309910     0.190324  keyword article
9    10  content_73c54f78c06a           213963     0.250324  keyword article

Columns used in the score: ['position_tier', 'avg_position', 'ctr', 'impressions_90d']
None of these are product flags (health_score, priority_score, ac

/tmp/ipykernel_2113/3888949605.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top10["ctr_gap_pct"] = top10["tier_ctr_benchmark"] - top10["ctr"]


**Weak picks:** 11,631 / 30,000 pages (38.8%) get flagged — that's not a short list, it's a
third of the catalog. A rule that flags a third of everything isn't doing much prioritizing; a
tighter `ctr_gap_pct` floor (not just >0) would sharpen it. `keyword article` pages are also
mildly over-represented in the flagged set (98.8%) versus their overall share of the data
(90.7%) — a small skew, likely because that content type dominates the top position tiers by
volume, not because the rule specifically favors it. Worth watching if content-type balance
matters for whoever reviews this queue.

I'm treating the 38.8% flag rate as acceptable rather than a flaw — this baseline's job is to surface candidates for a human to review, not make the final call, and missing a real CTR problem costs more than a reviewer skimming past a false positive. A tighter threshold could always be tuned later once someone's had a look at what the reviewers actually reject.

**Leakage check:** confirmed — the score uses only `position_tier`, `avg_position`, `ctr`, and
`impressions_90d`. No `health_score`/`priority_score`/`action_type`/refresh-tier product flags
(not shipped in this data anyway), no `trend_pct` or `is_declining_label`, no future-window
columns. `is_declining` is used only in Section 1 to *evaluate* the staleness claim, never as a
score input.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.